# Poster Pipeline: Attention-Shape Clustering with Sliding SAX

This notebook regenerates a compact, poster-ready output folder.  It is designed to present a credible snapshot rather than a final taxonomy: stable clusters are named descriptively, unstable clusters are marked provisional, and open robustness checks are written out explicitly.

**Main output:** `poster_output/`


## 0. Configuration

Edit only the paths and headline modeling choices in this cell. The rest of the notebook is meant to run top-to-bottom.


In [1]:
from __future__ import annotations

import json
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch

from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from scipy.signal import savgol_filter
from scipy.stats import spearmanr

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# PATHS -- edit for your local machine
# -----------------------------------------------------------------------------
DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
POSTER_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_08\poster_output")

# Optional market file. If not found, prediction outputs are written with clear notes.
SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
DJ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\DOWJONES_data.csv")
NASDAQ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\NASDAQ100_data.csv")
RUSSELL_PATH = Path(r"C:\Python\CSUREMM\shock_detection\RUSSELL2000_data.csv")

# Input pattern inside each query folder.
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02*.csv"

# Date window used for poster snapshot.
START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# Filter / preprocessing choices

MAX_ZERO_SHARE = 0.50          # drop if more than 50% of values are zero
MAX_MISSING_SHARE = 0.02       # drop if more than ~2% of days are missing
MAX_INTERPOLATE_GAP = 7        # only interpolate gaps up to 7 consecutive days

DENOISE_WINDOW = 9
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91

# SAX / clustering choices. Keep compact for poster; expand grid only if needed.
WINDOW_SIZE = 28
STEP_SIZE = 7
N_SEGMENTS = 8
ALPHABET_SIZE = 4
K_RANGE = range(2, 10)
FINAL_K = 9
CONSENSUS_BOOTSTRAPS = 80
CONSENSUS_SUBSAMPLE = 0.80
STABILITY_THRESHOLD = 0.60
RANDOM_STATE = 42

for sub in [
    "00_provenance", "01_methodology", "02_cluster_selection", "03_cluster_shapes",
    "04_prediction", "05_validity_checks", "06_robustness_open_items"
]:
    (POSTER_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Poster output:", POSTER_DIR)


Poster output: C:\Python\CSUREMM\output\sax_tests_july_08\poster_output


## 1. Small helper functions


In [2]:
def clean_term_from_folder(folder: Path) -> str:
    return folder.name.replace("_", " ").strip()


def find_value_column(df: pd.DataFrame) -> str:
    date_like = {"time", "date", "week"}
    candidates = [c for c in df.columns if c.lower() not in date_like]

    if not candidates:
        raise ValueError("No value column found.")

    numeric = [
        c for c in candidates
        if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors="coerce"))
    ]

    return numeric[0] if numeric else candidates[0]


def max_consecutive_missing(s: pd.Series) -> int:
    missing = s.isna()

    if not missing.any():
        return 0

    groups = (missing != missing.shift()).cumsum()

    return int(
        missing
        .groupby(groups)
        .sum()
        .max()
    )


def load_one_series(folder: Path) -> pd.Series:
    stitched_dir = folder / STITCHED_SUBDIR
    files = sorted(stitched_dir.glob(STITCHED_GLOB)) if stitched_dir.exists() else []

    if not files:
        raise FileNotFoundError("missing stitched file")

    path = files[-1]
    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"time", "date", "week"}),
        df.columns[0],
    )

    value_col = find_value_column(df)

    dates = pd.to_datetime(df[date_col], errors="coerce")
    values = pd.to_numeric(df[value_col], errors="coerce")

    s = pd.Series(
        values.values,
        index=dates,
        name=clean_term_from_folder(folder),
    )

    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")]
    s = s.sort_index()

    s = s.loc[START_DATE:END_DATE]

    full_index = pd.date_range(
        START_DATE,
        END_DATE,
        freq="D",
    )

    return s.reindex(full_index)


def build_panel(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    series = {}

    folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])

    expected_days = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    for folder in folders:
        term = clean_term_from_folder(folder)

        try:
            s = load_one_series(folder)

            observed_share = float(s.notna().mean())
            missing_share = float(s.isna().mean())
            zero_share = float((s.dropna() == 0).mean())
            longest_missing_gap = max_consecutive_missing(s)
            n_unique = int(s.nunique(dropna=True))

            reason = "kept"

            if zero_share > MAX_ZERO_SHARE:
                reason = "high_zero_share"
            elif missing_share > MAX_MISSING_SHARE:
                reason = "high_missing_share"
            elif longest_missing_gap > MAX_INTERPOLATE_GAP:
                reason = "long_missing_gap"
            elif n_unique <= 2:
                reason = "near_constant"

            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": int(s.notna().sum()),
                "observed_share": observed_share,
                "missing_share": missing_share,
                "zero_share": zero_share,
                "longest_missing_gap": longest_missing_gap,
                "n_unique": n_unique,
                "drop_reason": reason,
            })

            if reason == "kept":
                series[term] = s

        except Exception as e:
            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": 0,
                "observed_share": 0.0,
                "missing_share": 1.0,
                "zero_share": np.nan,
                "longest_missing_gap": np.nan,
                "n_unique": 0,
                "drop_reason": str(e),
            })

    panel = pd.DataFrame(series).sort_index()
    funnel = pd.DataFrame(rows).sort_values(["drop_reason", "term"])

    return panel, funnel


def interpolate_small_gaps(s: pd.Series) -> pd.Series:
    return (
        s.astype(float)
        .interpolate(
            method="time",
            limit=MAX_INTERPOLATE_GAP,
            limit_direction="both",
        )
    )


def denoise_series(s: pd.Series) -> pd.Series:
    x = s.astype(float)
    valid = x.dropna()

    if len(valid) < DENOISE_WINDOW or DENOISE_WINDOW % 2 == 0:
        return x

    filled = x.interpolate(
        method="time",
        limit_direction="both",
    )

    return pd.Series(
        savgol_filter(
            filled,
            DENOISE_WINDOW,
            DENOISE_POLYORDER,
        ),
        index=x.index,
        name=x.name,
    )


def detrend_series(s: pd.Series) -> pd.Series:
    trend = s.rolling(
        DETREND_WINDOW,
        center=True,
        min_periods=max(14, DETREND_WINDOW // 4),
    ).median()

    return s - trend


def robust_zscore(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    x = s.astype(float)

    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)

        if pd.isna(std) or std == 0:
            return pd.Series(
                np.zeros(len(x)),
                index=x.index,
                name=x.name,
            )

        return (
            (x - x.mean(skipna=True)) / std
        ).rename(x.name)

    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    mad_ratio = (
        mad / wide_spread
        if wide_spread > 0
        else 1.0
    )

    is_degenerate = mad_ratio < mad_ratio_threshold

    if not is_degenerate:
        return (
            0.6745 * (x - med) / mad
        ).rename(x.name)

    mad_floored = (
        max(mad, mad_floor_frac * wide_spread)
        if wide_spread > 0
        else mad
    )

    z = (
        0.6745 * (x - med) / mad_floored
    ).rename(x.name)

    return z.clip(
        lower=-clip_mad_multiple,
        upper=clip_mad_multiple,
    )


def preprocess_panel(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:

    stages = {}

    stages["filled"] = panel.apply(
        interpolate_small_gaps,
        axis=0,
    )

    stages["denoised"] = stages["filled"].apply(
        denoise_series,
        axis=0,
    )

    stages["detrended"] = stages["denoised"].apply(
        detrend_series,
        axis=0,
    )

    stages["normalized"] = stages["detrended"].apply(
        robust_zscore,
        axis=0,
    )

    return stages["normalized"], stages

## 2. Load, filter, and preprocess

This produces the filtering funnel used to justify `1203 → retained N` and a one-term preprocessing figure for the methods panel.


In [3]:
panel_raw, filtering_funnel = build_panel(
    DATA_DIR
)

filtering_funnel.to_csv(
    POSTER_DIR / "00_provenance" / "filtering_funnel.csv",
    index=False,
)

panel_norm, stages = preprocess_panel(panel_raw)
panel_norm = panel_norm.dropna(axis=1, how="all")

print("Raw retained panel:", panel_raw.shape)
print(
    filtering_funnel["drop_reason"]
    .value_counts(dropna=False)
    .to_string()
)

Raw retained panel: (1612, 847)
drop_reason
kept                  847
high_zero_share       352
high_missing_share      4


## 3. SAX representation

Sliding SAX converts each normalized series into a compact symbolic shape profile. This keeps the poster explanation simple: local windows → segment averages → symbols → frequency vector.


In [5]:
def gaussian_breakpoints(a: int) -> np.ndarray:
    return stats.norm.ppf(np.linspace(0, 1, a + 1)[1:-1])


def paa(x: np.ndarray, n_segments: int) -> np.ndarray:
    chunks = np.array_split(x, n_segments)
    return np.array([np.nanmean(c) for c in chunks])


def sax_word(window: np.ndarray, n_segments=N_SEGMENTS, alphabet_size=ALPHABET_SIZE) -> str:
    x = np.asarray(window, dtype=float)
    x = (x - np.nanmean(x)) / (np.nanstd(x) if np.nanstd(x) else 1)
    seg = paa(x, n_segments)
    bins = gaussian_breakpoints(alphabet_size)
    alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"[:alphabet_size]))
    return "".join(alphabet[np.searchsorted(bins, v)] for v in seg)


def sax_feature_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for term in panel.columns:
        x = panel[term].dropna().values
        words = []
        for start in range(0, max(0, len(x) - WINDOW_SIZE + 1), STEP_SIZE):
            w = x[start:start + WINDOW_SIZE]
            if np.isfinite(w).mean() >= 0.90:
                words.append(sax_word(w))
        counts = pd.Series(words).value_counts(normalize=True)
        counts.name = term
        rows.append(counts)
    X = pd.DataFrame(rows).fillna(0.0)
    X.index = panel.columns
    return X

X_sax = sax_feature_matrix(panel_norm)
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_sax), index=X_sax.index, columns=X_sax.columns)
print("SAX feature matrix:", X_scaled.shape)

# SAX explainer figure for poster.
term = "tsla"
s = panel_norm[term].dropna().iloc[:WINDOW_SIZE]
seg = paa(((s - s.mean()) / s.std()).values, N_SEGMENTS)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(range(len(s)), ((s - s.mean()) / s.std()).values, marker="o", linewidth=1.2, label="normalized window")
for i, val in enumerate(seg):
    lo = int(i * WINDOW_SIZE / N_SEGMENTS)
    hi = int((i + 1) * WINDOW_SIZE / N_SEGMENTS)
    ax.hlines(val, lo, hi - 1, linewidth=3)
ax.set_title(f"SAX example: {term} → {sax_word(s.values)}")
ax.set_xlabel("Days inside local window")
ax.set_ylabel("z-score")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(POSTER_DIR / "01_methodology" / "fig_sax_example.png", dpi=220)
plt.close(fig)


SAX feature matrix: (847, 25116)


## 4. Cluster selection and stability

This section compares candidate `k` values, estimates consensus stability, and draws the k=2→k=9 transition as a compact Sankey-style plot.


In [7]:
# -------------------------------------------------------------------------
# Efficient cluster-size selection
# -------------------------------------------------------------------------

FAST_KMEANS_N_INIT = 10
FAST_BATCH_SIZE = 512
FAST_CONSENSUS_BOOTSTRAPS = 25
FAST_CONSENSUS_SUBSAMPLE = 0.70
N_CONSENSUS_CANDIDATES = 3
SILHOUETTE_SAMPLE_SIZE = min(500, len(X_scaled))


def safe_zscore(s: pd.Series) -> pd.Series:
    sd = s.std(ddof=0)

    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s - s.mean()) / sd


def kmeans_labels(
    X: pd.DataFrame,
    k: int,
    seed: int = RANDOM_STATE,
    n_init: int = FAST_KMEANS_N_INIT,
) -> np.ndarray:

    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=seed,
        n_init=n_init,
        batch_size=FAST_BATCH_SIZE,
        reassignment_ratio=0.01,
    )

    return km.fit_predict(X)


def validation_table(X: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for k in K_RANGE:
        print(f"Validation k={k}")

        labels = kmeans_labels(X, k)

        sil = silhouette_score(
            X,
            labels,
            sample_size=SILHOUETTE_SAMPLE_SIZE,
            random_state=RANDOM_STATE,
        )

        rows.append({
            "k": k,
            "silhouette": sil,
            "davies_bouldin": davies_bouldin_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
        })

    val = pd.DataFrame(rows)

    val["silhouette_z"] = safe_zscore(val["silhouette"])
    val["davies_bouldin_z"] = -safe_zscore(val["davies_bouldin"])
    val["calinski_harabasz_z"] = safe_zscore(val["calinski_harabasz"])

    val["validation_score"] = val[
        [
            "silhouette_z",
            "davies_bouldin_z",
            "calinski_harabasz_z",
        ]
    ].mean(axis=1)

    return val


def candidate_k_from_validation(
    val: pd.DataFrame,
    n_candidates: int = N_CONSENSUS_CANDIDATES,
) -> list[int]:

    candidates = (
        val
        .sort_values("validation_score", ascending=False)
        .head(n_candidates)["k"]
        .astype(int)
        .tolist()
    )

    return sorted(candidates)


def consensus_matrix(
    X: pd.DataFrame,
    k: int,
    n_bootstraps: int = FAST_CONSENSUS_BOOTSTRAPS,
    subsample: float = FAST_CONSENSUS_SUBSAMPLE,
) -> np.ndarray:

    rng = np.random.default_rng(RANDOM_STATE)
    n = len(X)

    co = np.zeros((n, n), dtype=np.float32)
    seen = np.zeros((n, n), dtype=np.float32)
    idx = np.arange(n)

    sub_size = max(k + 1, int(subsample * n))

    for b in range(n_bootstraps):
        if b % 5 == 0:
            print(f"  consensus k={k}: bootstrap {b + 1}/{n_bootstraps}")

        sub = np.sort(
            rng.choice(
                idx,
                size=sub_size,
                replace=False,
            )
        )

        labs = kmeans_labels(
            X.iloc[sub],
            k,
            seed=RANDOM_STATE + b,
            n_init=5,
        )

        same = labs[:, None] == labs[None, :]

        co[np.ix_(sub, sub)] += same.astype(np.float32)
        seen[np.ix_(sub, sub)] += 1.0

    return np.divide(
        co,
        seen,
        out=np.zeros_like(co),
        where=seen > 0,
    )


def pac_score(
    C: np.ndarray,
    lo: float = 0.1,
    hi: float = 0.9,
) -> float:

    tri = C[np.triu_indices_from(C, k=1)]

    return float(
        ((tri > lo) & (tri < hi)).mean()
    )


def auto_select_k(
    val: pd.DataFrame,
    pac: pd.DataFrame,
) -> tuple[int, pd.DataFrame]:

    score_df = val.merge(pac, on="k", how="inner").copy()

    score_df["PAC_z"] = -safe_zscore(score_df["PAC"])

    score_df["auto_k_score"] = (
        0.50 * score_df["validation_score"]
        + 0.50 * score_df["PAC_z"]
    )

    best_score = score_df["auto_k_score"].max()
    score_sd = score_df["auto_k_score"].std(ddof=0)

    if pd.isna(score_sd) or score_sd == 0:
        score_sd = 0

    near_best = score_df[
        score_df["auto_k_score"] >= best_score - 0.25 * score_sd
    ].copy()

    selected_k = int(
        near_best
        .sort_values(["k", "auto_k_score"], ascending=[True, False])
        .iloc[0]["k"]
    )

    score_df["selected"] = score_df["k"].eq(selected_k)

    return selected_k, score_df


# -------------------------------------------------------------------------
# 1. Validation over full K_RANGE
# -------------------------------------------------------------------------

val = validation_table(X_scaled)

val.to_csv(
    POSTER_DIR / "02_cluster_selection" / "validation_metrics.csv",
    index=False,
)


# -------------------------------------------------------------------------
# 2. Consensus only for top candidate k values
# -------------------------------------------------------------------------

candidate_ks = candidate_k_from_validation(val)

print("Consensus candidate k values:", candidate_ks)

consensus_by_k = {}
pac_rows = []

for k in candidate_ks:
    C = consensus_matrix(X_scaled, k)
    consensus_by_k[k] = C

    pac_rows.append({
        "k": k,
        "PAC": pac_score(C),
    })

pac = pd.DataFrame(pac_rows)

pac.to_csv(
    POSTER_DIR / "02_cluster_selection" / "PAC_scores.csv",
    index=False,
)


# -------------------------------------------------------------------------
# 3. Auto-select final k
# -------------------------------------------------------------------------

FINAL_K, k_selection = auto_select_k(
    val=val,
    pac=pac,
)

k_selection.to_csv(
    POSTER_DIR / "02_cluster_selection" / "auto_k_selection.csv",
    index=False,
)

print("=" * 70)
print("Automatic cluster-size selection")
print("=" * 70)

print(
    k_selection
    .sort_values("auto_k_score", ascending=False)
    [
        [
            "k",
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz",
            "PAC",
            "validation_score",
            "PAC_z",
            "auto_k_score",
            "selected",
        ]
    ]
    .to_string(index=False)
)

print(f"\nSelected FINAL_K = {FINAL_K}")


# -------------------------------------------------------------------------
# 4. Final labels
# -------------------------------------------------------------------------

labels_final = pd.Series(
    kmeans_labels(
        X_scaled,
        FINAL_K,
        seed=RANDOM_STATE,
        n_init=FAST_KMEANS_N_INIT,
    ),
    index=X_scaled.index,
    name="cluster",
)


# -------------------------------------------------------------------------
# 5. Consensus CDF for selected k
# -------------------------------------------------------------------------

C_final = consensus_by_k[FINAL_K]

tri = C_final[np.triu_indices_from(C_final, k=1)]
xs = np.sort(tri)
ys = np.arange(1, len(xs) + 1) / len(xs)

fig, ax = plt.subplots(figsize=(5.8, 4.2))
ax.plot(xs, ys)
ax.set_title(f"Consensus CDF, selected k={FINAL_K}")
ax.set_xlabel("Pairwise consensus")
ax.set_ylabel("Cumulative share")
fig.tight_layout()

fig.savefig(
    POSTER_DIR / "02_cluster_selection" / "fig_consensus_CDF.png",
    dpi=220,
)

plt.close(fig)


# -------------------------------------------------------------------------
# 6. PAC curve for evaluated candidate k values
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.8, 4.2))
ax.plot(pac["k"], pac["PAC"], marker="o")
ax.axvline(FINAL_K, linestyle="--", linewidth=1)
ax.set_title("PAC curve over consensus-evaluated candidate k")
ax.set_xlabel("k")
ax.set_ylabel("PAC")
fig.tight_layout()

fig.savefig(
    POSTER_DIR / "02_cluster_selection" / "fig_PAC_curve.png",
    dpi=220,
)

plt.close(fig)


# -------------------------------------------------------------------------
# 7. Internal validation metrics over full K_RANGE
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.2, 4.4))

for col in ["silhouette", "davies_bouldin", "calinski_harabasz"]:
    z = safe_zscore(val[col])

    if col == "davies_bouldin":
        z = -z
        label = "Davies-Bouldin inverted"
    else:
        label = col

    ax.plot(
        val["k"],
        z,
        marker="o",
        label=label,
    )

ax.axvline(FINAL_K, linestyle="--", linewidth=1)
ax.set_title("Internal validation metrics, standardized")
ax.set_xlabel("k")
ax.set_ylabel("Better →")
ax.legend(frameon=False)

fig.tight_layout()

fig.savefig(
    POSTER_DIR / "02_cluster_selection" / "fig_validation_metrics.png",
    dpi=220,
)

plt.close(fig)

Validation k=2
Validation k=3


ValueError: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)

In [ ]:
COARSE_K = int(k_selection.sort_values("k").iloc[0]["k"])
FINAL_K = int(k_selection.loc[k_selection["selected"], "k"].iloc[0])

labels_coarse = pd.Series(
    kmeans_labels(X_scaled, COARSE_K),
    index=X_scaled.index,
    name=f"k{COARSE_K}",
)

labels_final = pd.Series(
    kmeans_labels(X_scaled, FINAL_K),
    index=X_scaled.index,
    name="cluster",
)


def plot_sankey_counts(
    left_labels: pd.Series,
    right_labels: pd.Series,
    path: Path,
    left_k: int,
    right_k: int,
):
    tab = pd.crosstab(left_labels, right_labels)

    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.axis("off")

    left_tot = tab.sum(axis=1)
    right_tot = tab.sum(axis=0)
    total = tab.values.sum()

    ly, ry = {}, {}

    y = 0
    for lab, cnt in left_tot.items():
        ly[lab] = (y, y + cnt / total)
        y += cnt / total + 0.03

    y = 0
    for lab, cnt in right_tot.items():
        ry[lab] = (y, y + cnt / total)
        y += cnt / total + 0.015

    left_used = {k: v[0] for k, v in ly.items()}
    right_used = {k: v[0] for k, v in ry.items()}

    for l in tab.index:
        for r in tab.columns:
            cnt = tab.loc[l, r]

            if cnt == 0:
                continue

            h = cnt / total
            y0, y1 = left_used[l], right_used[r]

            left_used[l] += h
            right_used[r] += h

            verts = [
                (0.15, y0),
                (0.42, y0),
                (0.58, y1),
                (0.85, y1),
                (0.85, y1 + h),
                (0.58, y1 + h),
                (0.42, y0 + h),
                (0.15, y0 + h),
                (0.15, y0),
            ]

            codes = [
                MplPath.MOVETO,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.LINETO,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.CLOSEPOLY,
            ]

            ax.add_patch(
                PathPatch(
                    MplPath(verts, codes),
                    alpha=0.25,
                    lw=0.5,
                )
            )

    for lab, (a, b) in ly.items():
        ax.text(
            0.05,
            (a + b) / 2,
            f"k={left_k} C{lab}\n{left_tot[lab]}",
            ha="right",
            va="center",
        )

    for lab, (a, b) in ry.items():
        ax.text(
            0.95,
            (a + b) / 2,
            f"k={right_k} C{lab}\n{right_tot[lab]}",
            ha="left",
            va="center",
        )

    ax.set_title(
        f"How coarse k={left_k} groups split into selected k={right_k} clusters"
    )

    fig.tight_layout()
    fig.savefig(path, dpi=220)
    plt.close(fig)


plot_sankey_counts(
    labels_coarse,
    labels_final,
    POSTER_DIR / "02_cluster_selection" / f"fig_sankey_k{COARSE_K}_to_k{FINAL_K}.png",
    left_k=COARSE_K,
    right_k=FINAL_K,
)

## 5. Cluster profiles: medoids, stable cores, and descriptive names

Cluster names describe curve shapes only. Stability is shown directly so the poster does not overclaim provisional groups.


In [ ]:
def summarize_cluster_stability(labels: pd.Series, C: np.ndarray) -> pd.DataFrame:
    rows = []
    arr = labels.values

    for cl in sorted(labels.unique()):
        ix = np.where(arr == cl)[0]

        if len(ix) <= 1:
            rows.append({
                "cluster": cl,
                "n_terms": len(ix),
                "cluster_stability": np.nan,
                "stability_mean": np.nan,
                "stability_sd": np.nan,
                "stability_iqr": np.nan,
            })
            continue

        sub = C[np.ix_(ix, ix)]
        pair_vals = sub[np.triu_indices_from(sub, k=1)]

        rows.append({
            "cluster": cl,
            "n_terms": len(ix),
            "cluster_stability": float(np.nanmedian(pair_vals)),
            "stability_mean": float(np.nanmean(pair_vals)),
            "stability_sd": float(np.nanstd(pair_vals)),
            "stability_iqr": float(
                np.nanpercentile(pair_vals, 75)
                - np.nanpercentile(pair_vals, 25)
            ),
        })

    return pd.DataFrame(rows)


def compute_term_stability(labels: pd.Series, C: np.ndarray) -> pd.DataFrame:
    rows = []

    for i, term in enumerate(labels.index):
        cl = labels.loc[term]
        mates = np.where(labels.values == cl)[0]
        mates = mates[mates != i]

        score = (
            float(np.nanmean(C[i, mates]))
            if len(mates) > 0
            else np.nan
        )

        rows.append({
            "term": term,
            "cluster": cl,
            "term_stability": score,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["cluster", "term_stability"], ascending=[True, False])
        .reset_index(drop=True)
    )


def compute_core_rules(
    labels: pd.Series,
    term_stability: pd.DataFrame,
) -> dict:
    stab = term_stability["term_stability"].dropna()
    cluster_sizes = labels.value_counts()

    stability_threshold = max(
        0.70,
        float(stab.quantile(0.75)),
    )

    min_core_terms = max(
        5,
        int(np.ceil(0.20 * cluster_sizes.median())),
    )

    core_top_frac = 0.50

    return {
        "stability_threshold": stability_threshold,
        "min_core_terms": min_core_terms,
        "core_top_frac": core_top_frac,
    }


def select_core_terms(
    term_stability: pd.DataFrame,
    rules: dict,
) -> pd.DataFrame:
    rows = []

    threshold = rules["stability_threshold"]
    min_core_terms = rules["min_core_terms"]
    core_top_frac = rules["core_top_frac"]

    for cl, g in term_stability.groupby("cluster"):
        g = g.sort_values("term_stability", ascending=False).copy()

        stable = g[g["term_stability"] >= threshold].copy()
        min_keep = min(min_core_terms, len(g))
        fallback_keep = max(
            min_keep,
            int(np.ceil(core_top_frac * len(g))),
        )

        if len(stable) >= min_keep:
            core = stable.copy()
            rule = f"term_stability >= {threshold:.3f}"
        else:
            core = g.head(fallback_keep).copy()
            rule = f"fallback_top_{core_top_frac:.0%}"

        core["core_rank"] = np.arange(1, len(core) + 1)
        core["core_rule"] = rule

        rows.append(core)

    return pd.concat(rows, axis=0).reset_index(drop=True)


def compute_medoids(X: pd.DataFrame, labels: pd.Series) -> pd.DataFrame:
    rows = []
    D = cosine_distances(X)

    for cl in sorted(labels.unique()):
        ix = np.where(labels.values == cl)[0]
        subD = D[np.ix_(ix, ix)]
        medoid_ix = ix[subD.mean(axis=1).argmin()]

        rows.append({
            "cluster": cl,
            "medoid_term": X.index[medoid_ix],
            "n_terms": len(ix),
        })

    return pd.DataFrame(rows)


def build_core_indices(
    panel: pd.DataFrame,
    core_terms: pd.DataFrame,
) -> pd.DataFrame:
    out = pd.DataFrame(index=panel.index)

    for cl, g in core_terms.groupby("cluster"):
        terms = [t for t in g["term"].tolist() if t in panel.columns]

        if terms:
            out[f"cluster_{cl}_core_index"] = panel[terms].mean(axis=1)

    return out


cluster_stability = summarize_cluster_stability(labels_final, C_final)

cluster_stability["stability_flag"] = np.where(
    cluster_stability["cluster_stability"] >= STABILITY_THRESHOLD,
    "stable",
    "provisional",
)

term_stability = compute_term_stability(labels_final, C_final)

core_rules = compute_core_rules(labels_final, term_stability)

core_terms = select_core_terms(term_stability, core_rules)

core_indices = build_core_indices(panel_norm, core_terms)

medoids = (
    compute_medoids(X_scaled, labels_final)
    .merge(cluster_stability, on=["cluster", "n_terms"], how="left")
)

core_summary = (
    core_terms
    .groupby("cluster")
    .agg(
        n_core_terms=("term", "count"),
        min_core_stability=("term_stability", "min"),
        mean_core_stability=("term_stability", "mean"),
        median_core_stability=("term_stability", "median"),
        core_rule=("core_rule", "first"),
    )
    .reset_index()
)

cluster_keywords = (
    core_terms
    .groupby("cluster")["term"]
    .apply(lambda x: ", ".join(x.head(20)))
    .reset_index(name="core_keywords")
)

cluster_summary = (
    medoids
    .merge(core_summary, on="cluster", how="left")
    .merge(cluster_keywords, on="cluster", how="left")
)

cluster_summary.to_csv(
    POSTER_DIR / "03_cluster_shapes" / "cluster_keywords.csv",
    index=False,
)

medoids.to_csv(
    POSTER_DIR / "03_cluster_shapes" / "cluster_medoids.csv",
    index=False,
)

term_stability.to_csv(
    POSTER_DIR / "03_cluster_shapes" / "stability_scores.csv",
    index=False,
)

core_terms.to_csv(
    POSTER_DIR / "03_cluster_shapes" / "stable_core_terms.csv",
    index=False,
)

core_indices.to_csv(
    POSTER_DIR / "03_cluster_shapes" / "stable_core_indices.csv",
)

pd.DataFrame([core_rules]).to_csv(
    POSTER_DIR / "03_cluster_shapes" / "core_selection_rules.csv",
    index=False,
)

print(
    cluster_summary[
        [
            "cluster",
            "stability_flag",
            "n_terms",
            "n_core_terms",
            "cluster_stability",
            "stability_sd",
            "medoid_term",
            "core_rule",
        ]
    ].to_string(index=False)
)

## 6. Prediction snapshot

This is intentionally stability-aware: every row carries cluster stability, and weak/null results are kept in the table rather than hidden.


In [ ]:
# -------------------------------------------------------------------------
# STEP 6: Prediction using stable-core cluster indices
# -------------------------------------------------------------------------

MARKET_PATHS = {
    "sp500": SP500_PATH,
    "dowjones": DJ_PATH,
    "nasdaq100": NASDAQ_PATH,
    "russell2000": RUSSELL_PATH,
}


def load_market_targets(path: Path, index_name: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"date", "time"}),
        df.columns[0],
    )

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.set_index(date_col).sort_index()

    names = {c.lower().replace(" ", "_"): c for c in df.columns}
    numeric_cols = df.select_dtypes("number").columns.tolist()

    if not numeric_cols:
        return pd.DataFrame()

    close_col = (
        names.get("adj_close")
        or names.get("adjusted_close")
        or names.get("close")
        or numeric_cols[0]
    )

    close = pd.to_numeric(df[close_col], errors="coerce")
    ret = close.pct_change()

    return pd.DataFrame({
        f"{index_name}_abs_return_1d": ret.abs(),
        f"{index_name}_abs_return_2d": ret.abs().rolling(2).sum(),
        f"{index_name}_abs_return_7d": ret.abs().rolling(7).sum(),
        f"{index_name}_return_1d": ret,
    })


def load_all_market_targets(market_paths: dict[str, Path]) -> pd.DataFrame:
    targets = []

    for index_name, path in market_paths.items():
        target = load_market_targets(path, index_name)

        if not target.empty:
            targets.append(target)
        else:
            print(f"Missing or unusable market file: {index_name} -> {path}")

    if not targets:
        return pd.DataFrame()

    return pd.concat(targets, axis=1).sort_index()


def directional_accuracy(actual: np.ndarray, pred: np.ndarray) -> float:
    actual_sign = np.sign(actual)
    pred_sign = np.sign(pred)

    valid = actual_sign != 0

    if valid.sum() == 0:
        return np.nan

    return float((actual_sign[valid] == pred_sign[valid]).mean())


def rolling_oos(X: pd.DataFrame, y: pd.Series, min_train: int = 500) -> dict:
    df = X.join(y.rename("target")).dropna()

    if len(df) < min_train + 60:
        return {
            "initial_train_n": min_train,
            "test_n": len(df),
            "oos_r2": np.nan,
            "oos_rmse": np.nan,
            "oos_corr": np.nan,
            "oos_directional_accuracy": np.nan,
            "baseline_directional_accuracy": np.nan,
            "delta_vs_baseline": np.nan,
            "train_r2": np.nan,
            "train_corr": np.nan,
        }

    preds = []
    actuals = []
    baselines = []

    for t in range(min_train, len(df)):
        train = df.iloc[:t]
        test = df.iloc[[t]]

        model = RidgeCV(
            alphas=np.logspace(-4, 4, 25)
        ).fit(
            train[X.columns],
            train["target"],
        )

        preds.append(model.predict(test[X.columns])[0])
        actuals.append(test["target"].iloc[0])
        baselines.append(train["target"].mean())

    pred = np.array(preds)
    actual = np.array(actuals)
    baseline = np.array(baselines)

    denom = np.sum((actual - baseline) ** 2)

    oos_r2 = (
        1 - np.sum((actual - pred) ** 2) / denom
        if denom > 0
        else np.nan
    )

    oos_rmse = np.sqrt(np.mean((actual - pred) ** 2))

    oos_corr = (
        np.corrcoef(actual, pred)[0, 1]
        if len(actual) > 2 and np.std(pred) > 0 and np.std(actual) > 0
        else np.nan
    )

    oos_dir = directional_accuracy(actual, pred)
    base_dir = directional_accuracy(actual, baseline)

    delta_vs_baseline = (
        oos_dir - base_dir
        if not pd.isna(oos_dir) and not pd.isna(base_dir)
        else np.nan
    )

    train = df.iloc[:min_train]

    fit = RidgeCV(
        alphas=np.logspace(-4, 4, 25)
    ).fit(
        train[X.columns],
        train["target"],
    )

    train_actual = train["target"]
    train_pred = fit.predict(train[X.columns])

    train_denom = np.sum((train_actual - train_actual.mean()) ** 2)

    train_r2 = (
        1 - np.sum((train_actual - train_pred) ** 2) / train_denom
        if train_denom > 0
        else np.nan
    )

    train_corr = (
        np.corrcoef(train_actual, train_pred)[0, 1]
        if np.std(train_pred) > 0 and np.std(train_actual) > 0
        else np.nan
    )

    return {
        "initial_train_n": min_train,
        "test_n": len(actual),
        "oos_r2": oos_r2,
        "oos_rmse": oos_rmse,
        "oos_corr": oos_corr,
        "oos_directional_accuracy": oos_dir,
        "baseline_directional_accuracy": base_dir,
        "delta_vs_baseline": delta_vs_baseline,
        "train_r2": train_r2,
        "train_corr": train_corr,
    }


targets = load_all_market_targets(MARKET_PATHS)

rows = []

if not targets.empty and not core_indices.empty:
    data = core_indices.join(targets, how="inner")

    for target in targets.columns:
        market_index = target.split("_")[0]
        target_type = target.replace(f"{market_index}_", "")

        for pred_col in core_indices.columns:
            cl = int(pred_col.split("_")[1])

            res = rolling_oos(
                data[[pred_col]],
                data[target],
            )

            cluster_row = cluster_summary.loc[
                cluster_summary["cluster"] == cl
            ].iloc[0]

            rows.append({
                "market_index": market_index,
                "target_type": target_type,
                "target_index": target,
                "cluster": cl,
                "predictor": pred_col,
                "n_terms": int(cluster_row["n_terms"]),
                "n_core_terms": int(cluster_row["n_core_terms"]),
                "cluster_stability": float(cluster_row["cluster_stability"]),
                "stability_flag": cluster_row["stability_flag"],
                **res,
            })

else:
    rows.append({
        "note": "Prediction not run: missing market targets or empty core indices."
    })


oos = pd.DataFrame(rows)

oos.to_csv(
    POSTER_DIR / "04_prediction" / "oos_metrics_by_cluster.csv",
    index=False,
)


if "oos_r2" in oos.columns:
    poster_table = oos[
        [
            "market_index",
            "target_type",
            "cluster",
            "oos_r2",
            "oos_directional_accuracy",
            "delta_vs_baseline",
            "stability_flag",
        ]
    ].copy()

    poster_table = poster_table.rename(columns={
        "market_index": "index",
        "target_type": "target",
        "cluster": "C",
        "oos_r2": "oos_R2",
        "oos_directional_accuracy": "oos_dir_acc",
        "delta_vs_baseline": "delta_vs_base",
        "stability_flag": "stability",
    })

    poster_table.to_csv(
        POSTER_DIR / "04_prediction" / "poster_prediction_table.csv",
        index=False,
    )


if "oos_r2" in oos.columns and oos["oos_r2"].notna().any():
    plot_df = oos.dropna(subset=["oos_r2"]).copy()

    plot_df["label"] = plot_df.apply(
        lambda r: (
            f"{r.market_index} / C{int(r.cluster)}"
            + ("*" if r.stability_flag == "provisional" else "")
        ),
        axis=1,
    )

    top = (
        plot_df
        .sort_values("oos_r2", ascending=False)
        .head(20)
        .sort_values("oos_r2")
    )

    fig, ax = plt.subplots(figsize=(8.2, 5.6))
    ax.barh(top["label"], top["oos_r2"])
    ax.axvline(0, linewidth=1)

    ax.set_title(
        "Poster snapshot: OOS R² using stable-core cluster indices\n"
        "(* = provisional cluster)"
    )
    ax.set_xlabel("oos R²")

    fig.tight_layout()

    fig.savefig(
        POSTER_DIR / "04_prediction" / "fig_oos_comparison.png",
        dpi=220,
    )

    plt.close(fig)

else:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axis("off")
    ax.text(
        0.02,
        0.6,
        "Prediction plot not generated: missing targets or insufficient overlap.",
    )

    fig.savefig(
        POSTER_DIR / "04_prediction" / "fig_oos_comparison.png",
        dpi=220,
    )

    plt.close(fig)

NameError: name 'SP500_PATH' is not defined

## 7. Validity checks and robustness placeholders

These outputs make the open questions explicit: resampling stability, event concentration, missingness threshold sensitivity, history length sensitivity, and stitching calibration.


In [ ]:
# -------------------------------------------------------------------------
# STEP 7: Validity checks and robustness open items
# -------------------------------------------------------------------------

perm_rows = []
N_PERMUTATIONS = 100

if not targets.empty and "oos_r2" in oos.columns and oos["oos_r2"].notna().any():

    observed = oos["oos_r2"].max()

    observed_row = (
        oos
        .dropna(subset=["oos_r2"])
        .sort_values("oos_r2", ascending=False)
        .iloc[0]
    )

    for b in range(N_PERMUTATIONS):
        shuffled = (
            labels_final
            .sample(frac=1, random_state=RANDOM_STATE + b)
            .set_axis(labels_final.index)
        )

        shuffled_term_stability = compute_term_stability(
            shuffled,
            C_final,
        )

        shuffled_core_terms = select_core_terms(
            shuffled_term_stability,
            core_rules,
        )

        shuffled_indices = build_core_indices(
            panel_norm,
            shuffled_core_terms,
        )

        best = -np.inf

        for target in targets.columns:
            for pred_col in shuffled_indices.columns:
                res = rolling_oos(
                    shuffled_indices[[pred_col]],
                    targets[target],
                )

                if not pd.isna(res["oos_r2"]):
                    best = max(best, res["oos_r2"])

        perm_rows.append({
            "iteration": b,
            "best_oos_r2": best,
            "observed_best_oos_r2": observed,
            "observed_target_index": observed_row["target_index"],
            "observed_cluster": observed_row["cluster"],
        })

    empirical_p = (
        1 + sum(r["best_oos_r2"] >= observed for r in perm_rows)
    ) / (
        1 + len(perm_rows)
    )

    pd.DataFrame([{
        "observed_best_oos_r2": observed,
        "observed_target_index": observed_row["target_index"],
        "observed_cluster": observed_row["cluster"],
        "empirical_pvalue": empirical_p,
        "note": "Permutation uses shuffled labels and stable-core index reconstruction.",
    }]).to_csv(
        POSTER_DIR / "05_validity_checks" / "empirical_pvalues.csv",
        index=False,
    )

else:
    perm_rows = [{
        "note": "Permutation check not run: missing prediction results."
    }]

    pd.DataFrame(perm_rows).to_csv(
        POSTER_DIR / "05_validity_checks" / "empirical_pvalues.csv",
        index=False,
    )


perm_df = pd.DataFrame(perm_rows)

fig, ax = plt.subplots(figsize=(6.5, 4.2))

if "best_oos_r2" in perm_df.columns:
    ax.hist(perm_df["best_oos_r2"].dropna(), bins=20, alpha=0.8)
    ax.axvline(
        perm_df["observed_best_oos_r2"].iloc[0],
        linestyle="--",
        linewidth=2,
    )
    ax.set_title("Permutation distribution of best oos R²")
    ax.set_xlabel("Best shuffled oos R²")
    ax.set_ylabel("Count")
else:
    ax.axis("off")
    ax.text(0.02, 0.6, perm_df.iloc[0, 0])

fig.tight_layout()

fig.savefig(
    POSTER_DIR / "05_validity_checks" / "fig_permutation_distribution.png",
    dpi=220,
)

plt.close(fig)


event_rows = []

for cl in sorted(labels_final.unique()):
    members = labels_final[labels_final == cl].index
    z = panel_norm[members]

    high = (
        z
        .gt(z.quantile(0.95))
        .sum()
        .sort_values(ascending=False)
    )

    event_rows.append({
        "cluster": cl,
        "n_terms": len(members),
        "top_term": high.index[0] if len(high) else None,
        "top_term_event_share": (
            float(high.iloc[0] / high.sum())
            if high.sum()
            else np.nan
        ),
        "top_5_event_share": (
            float(high.head(5).sum() / high.sum())
            if high.sum()
            else np.nan
        ),
    })

pd.DataFrame(event_rows).to_csv(
    POSTER_DIR / "05_validity_checks" / "event_concentration.csv",
    index=False,
)


pd.DataFrame([
    {
        "setting": "main",
        "min_obs_share": MIN_OBS_SHARE,
        "n_retained": panel_norm.shape[1],
        "status": "completed",
    },
    {
        "setting": "looser_missingness",
        "min_obs_share": 0.70,
        "n_retained": np.nan,
        "status": "planned",
    },
    {
        "setting": "stricter_missingness",
        "min_obs_share": 0.90,
        "n_retained": np.nan,
        "status": "planned",
    },
]).to_csv(
    POSTER_DIR / "06_robustness_open_items" / "missingness_threshold_sensitivity.csv",
    index=False,
)


pd.DataFrame([
    {
        "window": "full history",
        "start": START_DATE,
        "end": END_DATE,
        "status": "completed",
    },
    {
        "window": "shorter / rolling history",
        "start": np.nan,
        "end": np.nan,
        "status": "planned",
    },
]).to_csv(
    POSTER_DIR / "06_robustness_open_items" / "history_length_sensitivity.csv",
    index=False,
)


pd.DataFrame([
    {
        "check": "stitched overlap ratios",
        "status": "audit before final claims",
    },
    {
        "check": "abnormal post-stitching ranges",
        "status": "partly handled by detrending and robust z-score",
    },
]).to_csv(
    POSTER_DIR / "06_robustness_open_items" / "stitching_calibration_check.csv",
    index=False,
)

## 8. Poster claims map

This writes a one-page README connecting each generated output to the poster claim it can support.


In [ ]:
claims = f"""# Poster claims map

This folder is for poster preparation, instead of the final analysis.

## 00_provenance/filtering_funnel.csv
Supports: transparent sample construction and drop reasons. Use this for the 1203 → retained-N funnel.

## 01_methodology/fig_example_preprocessing.png
Supports: raw Google Trends series are gap-filled only over short gaps, denoised, detrended, and robustly normalized before shape clustering.

## 01_methodology/fig_sax_example.png
Supports: local attention curves are transformed into SAX words, then clustered by symbolic shape profiles.

## 02_cluster_selection/fig_consensus_CDF.png and fig_PAC_curve.png
Supports: cluster stability is measured directly rather than assumed.

## 02_cluster_selection/fig_validation_metrics.png
Supports: k selection uses multiple internal metrics; no single metric is treated as definitive.

## 02_cluster_selection/fig_sankey_k2_to_k9.png
Supports: the k=2 versus k=9 issue is treated as a nesting / resolution question, not hidden.

## 03_cluster_shapes/cluster_profiles.pdf
Supports: descriptive cluster archetypes. Medoids are shown with faint stable-member curves to avoid cherry-picking.

## 03_cluster_shapes/cluster_keywords.csv and stability_scores.csv
Supports: cluster names are descriptive and stability-aware. Clusters below stability threshold {STABILITY_THRESHOLD:.2f} should be marked provisional.

## 04_prediction/oos_metrics_by_cluster.csv
Supports: predictive-power snapshot across cluster indices and market targets. Rows include train/test metrics and stability scores.

## 04_prediction/stable_core_prediction.csv
Supports: stripped-down comparison for the stable core of cluster 2, if available.

## 05_validity_checks/*
Supports: placebo/permutation and event-concentration checks. These are safeguards against overclaiming.

## 06_robustness_open_items/*
Supports: explicit next steps: missingness threshold sensitivity, history-length sensitivity, and stitching calibration checks.

Suggested poster wording: “We identify stable and provisional attention-shape archetypes; ongoing robustness checks test sensitivity to history length, missing-data filters, and stitching calibration.”
"""
